In [4]:
import torch
import torchvision
import torchvision.transforms as transforms

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt
import numpy as np

In [5]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


In [6]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.tanh(self.conv1(x)))
        x = self.pool(torch.tanh(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [8]:
def train_model(optimizer, criterion, epochs):
  for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

  print('Finished Training')



In [9]:
def predict_test():
  correct = 0
  total = 0
  # since we're not training, we don't need to calculate the gradients for our outputs
  with torch.no_grad():
      for data in testloader:
          images, labels = data
          # calculate outputs by running images through the network
          outputs = net(images)
          # the class with the highest energy is what we choose as prediction
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

  print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

### Use SGD as Optimizer



In [10]:
criterion = nn.CrossEntropyLoss()
optimizer_sgd = optim.SGD(net.parameters(), lr=0.0001)
train_model(optimizer_sgd, criterion, 5)
predict_test()

[1,  2000] loss: 2.307
[1,  4000] loss: 2.305
[1,  6000] loss: 2.305
[1,  8000] loss: 2.302
[1, 10000] loss: 2.300
[1, 12000] loss: 2.300
[2,  2000] loss: 2.296
[2,  4000] loss: 2.291
[2,  6000] loss: 2.287
[2,  8000] loss: 2.282
[2, 10000] loss: 2.276
[2, 12000] loss: 2.267
[3,  2000] loss: 2.254
[3,  4000] loss: 2.243
[3,  6000] loss: 2.227
[3,  8000] loss: 2.216
[3, 10000] loss: 2.196
[3, 12000] loss: 2.180
[4,  2000] loss: 2.166
[4,  4000] loss: 2.152
[4,  6000] loss: 2.141
[4,  8000] loss: 2.127
[4, 10000] loss: 2.122
[4, 12000] loss: 2.111
[5,  2000] loss: 2.101
[5,  4000] loss: 2.085
[5,  6000] loss: 2.081
[5,  8000] loss: 2.067
[5, 10000] loss: 2.063
[5, 12000] loss: 2.053
Finished Training
Accuracy of the network on the 10000 test images: 25 %


### Use ADAM as Optimizer


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer_adam = optim.Adam(net.parameters(), lr=0.0001)
train_model(optimizer_adam, criterion, 5)
predict_test()

[1,  2000] loss: 1.940
[1,  4000] loss: 1.783
[1,  6000] loss: 1.666
[1,  8000] loss: 1.595
[1, 10000] loss: 1.555
[1, 12000] loss: 1.504
[2,  2000] loss: 1.464
[2,  4000] loss: 1.459
[2,  6000] loss: 1.443
[2,  8000] loss: 1.411
[2, 10000] loss: 1.401
[2, 12000] loss: 1.391
[3,  2000] loss: 1.361
[3,  4000] loss: 1.358
[3,  6000] loss: 1.348
[3,  8000] loss: 1.341
[3, 10000] loss: 1.318
[3, 12000] loss: 1.301
[4,  2000] loss: 1.287
[4,  4000] loss: 1.258
[4,  6000] loss: 1.286
[4,  8000] loss: 1.279
[4, 10000] loss: 1.258
[4, 12000] loss: 1.245
[5,  2000] loss: 1.214
[5,  4000] loss: 1.245
[5,  6000] loss: 1.196
[5,  8000] loss: 1.204
[5, 10000] loss: 1.198
[5, 12000] loss: 1.212
Finished Training
Accuracy of the network on the 10000 test images: 56 %
